# MarineHydro.jl tutorial

Heave radiation and diffraction of a floating sphere in linear potential flow, then derivatives of the hydrodynamic coefficients with respect to radius and frequency.

The problem types follow Capytaine: a `FloatingBody`, a `RadiationProblem` (body, $\omega$) or `DiffractionProblem` (body, $\omega$, heading $\beta$), and `solve`. The formulation is `DirectBEM` or `IndirectBEM`, with Wu's Green function by default or Delhommeau if you pass `"ExactGuevelDelhommeau"`. Added mass, damping, Froude–Krylov, diffraction, and excitation come from the result. A list of frequencies is `hydrodynamic_coefficients`.

Differentiate the same functions with DifferentiationInterface (`AutoForwardDiff` or Enzyme reverse). Capytaine only builds the mesh; `fd_mesh_rules!` finite-differences geometry so Dual numbers never enter Python. (Number of panel stay fixed because of this)


In [ ]:
using Pkg
mh_root = get(ENV, "MARINEHYDRO_ROOT", @__DIR__)
Pkg.activate(mh_root)
Pkg.instantiate()
using MarineHydro
using PyCall
using StaticArrays
using DifferentiationInterface
import DifferentiationInterface as DI
using Enzyme
using Printf

cpt = pyimport("capytaine")

function sphere_smesh(r)
    StaticArraysMesh(cpt.mesh_sphere(
        name="sphere", radius=r, center=(0.0, 0.0, 0.0),
        resolution=(6, 6)).immersed_part())
end
fd_mesh_rules!(sphere_smesh)

sphere_body(r) = FloatingBody(sphere_smesh(r), [:Heave], "sphere")

const formulation = DirectBEM()                # default: direct + Wu
const r0 = 1.0
const ω0 = 1.03
const β0 = 0.0
const ωs = [0.8, 1.03, 1.4]
const fwd = AutoForwardDiff()
const rev = AutoEnzyme(; mode=Enzyme.set_runtime_activity(Enzyme.Reverse))

println("MarineHydro at ", mh_root)
println(typeof(sphere_smesh(r0)), "  nfaces = ", sphere_smesh(r0).nfaces)


## 1. One radiation solve

`DirectBEM()` holds the Green functions. `RadiationProblem(body, ω)` fills in \(k\)
and the single radiating dof.


In [ ]:
radiation_solve(r, ω, form=formulation) = solve(RadiationProblem(sphere_body(r), ω), form)

sol = radiation_solve(r0, ω0)
@printf("A₃₃ = %.6f\nB₃₃ = %.6f\n", added_mass(sol).Heave, radiation_damping(sol).Heave)


## 2. DifferentiationInterface — the simple pattern

One active argument. Freeze the rest as `const`. Same `f` for ForwardDiff and Enzyme.


In [ ]:
added_mass_r(r) = added_mass(radiation_solve(r, ω0)).Heave
added_mass_ω(ω) = added_mass(radiation_solve(r0, ω)).Heave

dA_dr  = DI.derivative(added_mass_r, fwd, r0)
dA_dr′ = DI.derivative(added_mass_r, rev, r0)   # first Enzyme call compiles
dA_dω  = DI.derivative(added_mass_ω, fwd, ω0)
dA_dω′ = DI.derivative(added_mass_ω, rev, ω0)

@printf("dA₃₃/dr   ForwardDiff %.6f   Enzyme %.6f\n", dA_dr, dA_dr′)
@printf("dA₃₃/dω   ForwardDiff %.6f   Enzyme %.6f\n", dA_dω, dA_dω′)


Timing comparison (compile already done above). `report_derivative` is optional —
day-to-day use is the four `DI.derivative` lines.


In [ ]:
_err(e) = (s = sprint(showerror, e); n = min(length(s), 180); n < length(s) ? s[1:n] * "…" : s)

function report_derivative(name, f, x)
    println(name, " at x = ", x)
    ref = NaN
    for (label, backend) in (("ForwardDiff", fwd), ("Enzyme reverse", rev))
        try
            t = @elapsed d = DI.derivative(f, backend, x)
            d = Float64(d)
            isnan(ref) && (ref = d)
            rel = isfinite(ref) && ref != 0 ? abs(d - ref) / abs(ref) : NaN
            @printf("  %-16s  %.6f   %.3f s   rel = %.3e\n", label, d, t, rel)
        catch e
            println("  ", rpad(label, 16), "  FAILED: ", _err(e))
        end
    end
end

report_derivative("dA₃₃/dr", added_mass_r, r0)
report_derivative("dA₃₃/dω", added_mass_ω, ω0)


## 3. Diffraction — primal and the same DI pattern

`DiffractionProblem(body, ω; beta)`. Outputs are complex forces; AD needs Re or Im.


In [ ]:
diffraction_solve(r, ω, form=formulation) =
    solve(DiffractionProblem(sphere_body(r), ω; beta=β0), form)

dsol = diffraction_solve(r0, ω0)
Fd = diffraction_force(dsol).Heave
Fk = froude_krylov_force(dsol).Heave
Fe = excitation_force(dsol).Heave
@printf("F_D    %.6f %+.6fim\n", real(Fd), imag(Fd))
@printf("F_FK   %.6f %+.6fim\n", real(Fk), imag(Fk))
@printf("F_ex   %.6f %+.6fim\n", real(Fe), imag(Fe))

Fex_r(r) = real(excitation_force(diffraction_solve(r, ω0)).Heave)
Fex_ω(ω) = real(excitation_force(diffraction_solve(r0, ω)).Heave)

@printf("dRe(F_ex)/dr   ForwardDiff %.6f   Enzyme %.6f\n",
    DI.derivative(Fex_r, fwd, r0), DI.derivative(Fex_r, rev, r0))
@printf("dRe(F_ex)/dω   ForwardDiff %.6f   Enzyme %.6f\n",
    DI.derivative(Fex_ω, fwd, ω0), DI.derivative(Fex_ω, rev, ω0))


## 4. Many frequencies — one `solve` of a vector of problems

`hydrodynamic_coefficients` builds the problems and returns a NamedTuple of real
arrays (the differentiable payload). Label with
`label_hydrodynamic_coefficients` *after* AD, not through it
(`create_DimStack` remains as a deprecated alias).


In [ ]:
params_rad = (wave_frequencies=ωs, radiating_dofs=[:Heave])
params_all = (wave_frequencies=ωs, radiating_dofs=[:Heave], wave_directions=[β0])

data = hydrodynamic_coefficients(sphere_body(r0), params_all, formulation)
println("A₃₃(ω)  = ", vec(data.added_mass))
println("B₃₃(ω)  = ", vec(data.radiation_damping))
println("F_ex(ω) = ", vec(data.excitation_force))


### Jacobian of the whole A($\omega)$ curve

- **wrt radius** (one geometry parameter → a vector of \(A\) at each \(\omega\)): `jacobian` on \(p=[r]\)
- **wrt the frequency vector:** `jacobian` of \(A(\omega_s)\) w.r.t. \(\omega_s\) (nearly diagonal)
- Enzyme reverse on a **scalar** reduction (`sum(A)`), then `gradient` on \(\omega_s\)


In [ ]:
A_of_r(r) = vec(hydrodynamic_coefficients(sphere_body(r), params_rad, formulation).added_mass)
A_of_ωs(ωvec) = vec(hydrodynamic_coefficients(sphere_body(r0),
    (wave_frequencies=ωvec, radiating_dofs=[:Heave]), formulation).added_mass)
sumA_r(r) = sum(A_of_r(r))
sumA_ωs(ωvec) = sum(A_of_ωs(ωvec))

J_r = DI.jacobian(p -> A_of_r(p[1]), fwd, [r0])          # size (nω, 1)
J_ω = DI.jacobian(A_of_ωs, fwd, copy(ωs))                # size (nω, nω)
g_r = DI.derivative(sumA_r, rev, r0)                     # Enzyme: d(∑A)/dr
g_ω = DI.gradient(sumA_ωs, rev, copy(ωs))                # Enzyme: d(∑A)/dωᵢ

println("A(ω)           ", A_of_r(r0))
println("∂A/∂r          ", vec(J_r))
println("∂Aᵢ/∂ωⱼ        "); display(J_ω)
@printf("Enzyme d(∑A)/dr     %.6f   (ForwardDiff %.6f)\n",
    g_r, DI.derivative(sumA_r, fwd, r0))
println("Enzyme d(∑A)/dω     ", g_ω)
println("ForwardDiff d(∑A)/dω ", DI.gradient(sumA_ωs, fwd, copy(ωs)))


## 5. Switch Green function and direct / indirect

The formulation is the method. Same `solve(prob, formulation)` / `hydrodynamic_coefficients(..., formulation)`.


In [ ]:
formulations = (
    "direct Wu"          => DirectBEM(),
    "indirect Wu"        => IndirectBEM(),
    "direct Delhommeau"  => DirectBEM("ExactGuevelDelhommeau"),
    "indirect Delhommeau"=> IndirectBEM("ExactGuevelDelhommeau"),
)

println("primal at r=$(r0), ω=$(ω0)")
@printf("  %-22s  %12s  %12s  %12s\n", "formulation", "A₃₃", "B₃₃", "Re(F_ex)")
for (name, form) in formulations
    A = added_mass(radiation_solve(r0, ω0, form)).Heave
    B = radiation_damping(radiation_solve(r0, ω0, form)).Heave
    Fe = real(excitation_force(diffraction_solve(r0, ω0, form)).Heave)
    @printf("  %-22s  %12.4f  %12.4f  %12.4f\n", name, A, B, Fe)
end


ForwardDiff through each formulation (Dual path). Enzyme reverse is shown on the
default `DirectBEM()` only — each new `formulation` is a fresh Enzyme compile.


In [ ]:
A_r_form(r, form) = added_mass(radiation_solve(r, ω0, form)).Heave
A_ω_form(ω, form) = added_mass(radiation_solve(r0, ω, form)).Heave
Fex_r_form(r, form) = real(excitation_force(diffraction_solve(r, ω0, form)).Heave)

println("ForwardDiff derivatives")
@printf("  %-22s  %14s  %14s  %14s\n", "formulation", "dA/dr", "dA/dω", "dRe(F_ex)/dr")
for (name, form) in formulations
    dAdr = DI.derivative(r -> A_r_form(r, form), fwd, r0)
    dAdω = DI.derivative(w -> A_ω_form(w, form), fwd, ω0)
    dFdr = DI.derivative(r -> Fex_r_form(r, form), fwd, r0)
    @printf("  %-22s  %14.4f  %14.4f  %14.4f\n", name, dAdr, dAdω, dFdr)
end

println("\nmany-ω A(ω) under each formulation (primal)")
for (name, form) in formulations
    Aω = vec(hydrodynamic_coefficients(sphere_body(r0), params_rad, form).added_mass)
    println("  ", rpad(name, 22), "  ", Aω)
end


## 6. Forward speed

Nonzero forward speed needs the indirect method (`IndirectBEM`), and the problem takes
the heading and wavenumber explicitly. The $\partial\phi/\partial x$ correction now runs on
the vectorized `StaticArraysMesh` kernels, and the whole solve stays differentiable
(ForwardDiff and Enzyme reverse).

In [ ]:
const U0 = 0.2   # forward speed [m/s]

radiation_solve_U(r, ω, U) = solve(
    RadiationProblem(sphere_body(r), ω, β0, compute_wavenumber(ω), U, :Heave, [:Heave]),
    IndirectBEM())

solU = radiation_solve_U(r0, ω0, U0)
@printf("A₃₃(U=%.1f) = %.6f   B₃₃(U=%.1f) = %.6f\n",
    U0, added_mass(solU).Heave, U0, radiation_damping(solU).Heave)

A_U_ω(ω) = added_mass(radiation_solve_U(r0, ω, U0)).Heave
A_U_r(r) = added_mass(radiation_solve_U(r, ω0, U0)).Heave

@printf("dA₃₃/dω  ForwardDiff %.6f   Enzyme %.6f\n",
    DI.derivative(A_U_ω, fwd, ω0), DI.derivative(A_U_ω, rev, ω0))
@printf("dA₃₃/dr  ForwardDiff %.6f   Enzyme %.6f\n",
    DI.derivative(A_U_r, fwd, r0), DI.derivative(A_U_r, rev, r0))

## Notes

```julia
fwd = AutoForwardDiff()
rev = AutoEnzyme(; mode=Enzyme.set_runtime_activity(Enzyme.Reverse))
f(r) = added_mass(solve(RadiationProblem(sphere_body(r), ω0), DirectBEM())).Heave
DI.derivative(f, fwd, r0)
DI.derivative(f, rev, r0)
```

| | |
|---|---|
| Green function | `DirectBEM()` Wu, `DirectBEM("ExactGuevelDelhommeau")` |
| Formulation | `DirectBEM` vs `IndirectBEM` |
| Many \(\omega\) | `hydrodynamic_coefficients(body, (wave_frequencies=ωs, radiating_dofs=[:Heave], wave_directions=[β]), formulation)` |
| Forward speed | `RadiationProblem(body, ω, β, k, U, dof, dofs)` + `IndirectBEM()` |
| Scalar \(r,\omega\) | `DI.derivative` |
| Vector \(\omega_s\) or \(A(\omega)\) | `DI.gradient` / `DI.jacobian` |
| Label axes | `label_hydrodynamic_coefficients(data, params, body)` after AD (`create_DimStack` is a deprecated alias) |
